In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, classification_report
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
import lightgbm as lgb

app = pd.read_csv('../data/processed/application_train_features.csv')
print("Shape:", app.shape)

Shape: (307507, 347)


In [3]:
X = app.drop(columns=['SK_ID_CURR', 'TARGET'])
y = app['TARGET']

print("X shape:", X.shape)
print("y distribution:\n", y.value_counts(normalize=True) * 100)

X shape: (307507, 345)
y distribution:
 TARGET
0    91.927013
1     8.072987
Name: proportion, dtype: float64


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train target %:", y_train.mean() * 100)
print("Test target %:", y_test.mean() * 100)

Train shape: (246005, 345)
Test shape: (61502, 345)
Train target %: 8.073006646206379
Test target %: 8.07290819810738


In [5]:
# Logistic Regression can't handle NaN — fill with median for numeric columns
X_train_filled = X_train.fillna(X_train.median(numeric_only=True))
X_test_filled = X_test.fillna(X_train.median(numeric_only=True))  # use TRAIN median to avoid leakage

print("Remaining NaNs in train:", X_train_filled.isnull().sum().sum())

Remaining NaNs in train: 0


In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_filled)
X_test_scaled = scaler.transform(X_test_filled)

log_reg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

y_pred_lr = log_reg.predict(X_test_scaled)
y_proba_lr = log_reg.predict_proba(X_test_scaled)[:, 1]

print("Logistic Regression Results:")
print("ROC-AUC:", roc_auc_score(y_test, y_proba_lr))
print(classification_report(y_test, y_pred_lr))

Logistic Regression Results:
ROC-AUC: 0.7638284732608601
              precision    recall  f1-score   support

           0       0.96      0.70      0.81     56537
           1       0.17      0.69      0.27      4965

    accuracy                           0.70     61502
   macro avg       0.57      0.70      0.54     61502
weighted avg       0.90      0.70      0.77     61502



In [7]:
rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_filled, y_train)  # tree models handle unscaled data fine, and can take NaN-filled data

y_pred_rf = rf.predict(X_test_filled)
y_proba_rf = rf.predict_proba(X_test_filled)[:, 1]

print("Random Forest Results:")
print("ROC-AUC:", roc_auc_score(y_test, y_proba_rf))
print(classification_report(y_test, y_pred_rf))

Random Forest Results:
ROC-AUC: 0.7507524780223508
              precision    recall  f1-score   support

           0       0.96      0.73      0.83     56537
           1       0.17      0.64      0.27      4965

    accuracy                           0.72     61502
   macro avg       0.57      0.69      0.55     61502
weighted avg       0.90      0.72      0.78     61502



In [8]:
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=(y_train.value_counts()[0] / y_train.value_counts()[1]),  # handles imbalance
    eval_metric='auc',
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train_filled, y_train)

y_pred_xgb = xgb_model.predict(X_test_filled)
y_proba_xgb = xgb_model.predict_proba(X_test_filled)[:, 1]

print("XGBoost Results:")
print("ROC-AUC:", roc_auc_score(y_test, y_proba_xgb))
print(classification_report(y_test, y_pred_xgb))

XGBoost Results:
ROC-AUC: 0.7782423744427025
              precision    recall  f1-score   support

           0       0.96      0.75      0.84     56537
           1       0.19      0.67      0.29      4965

    accuracy                           0.74     61502
   macro avg       0.58      0.71      0.57     61502
weighted avg       0.90      0.74      0.80     61502



In [10]:
import re

# Sanitize column names for LightGBM compatibility
X_train_lgb = X_train_filled.copy()
X_test_lgb = X_test_filled.copy()

X_train_lgb.columns = [re.sub(r'[^A-Za-z0-9_]+', '_', col) for col in X_train_lgb.columns]
X_test_lgb.columns = [re.sub(r'[^A-Za-z0-9_]+', '_', col) for col in X_test_lgb.columns]

# Confirm no duplicate column names were created by the sanitization
print("Duplicate columns after sanitizing:", X_train_lgb.columns[X_train_lgb.columns.duplicated()].tolist())

Duplicate columns after sanitizing: []


In [12]:
lgb_model = lgb.LGBMClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
lgb_model.fit(X_train_lgb, y_train)

y_pred_lgb = lgb_model.predict(X_test_lgb)
y_proba_lgb = lgb_model.predict_proba(X_test_lgb)[:, 1]

print("LightGBM Results:")
print("ROC-AUC:", roc_auc_score(y_test, y_proba_lgb))
print(classification_report(y_test, y_pred_lgb))


LightGBM Results:
ROC-AUC: 0.778279361512511
              precision    recall  f1-score   support

           0       0.96      0.73      0.83     56537
           1       0.18      0.69      0.29      4965

    accuracy                           0.73     61502
   macro avg       0.57      0.71      0.56     61502
weighted avg       0.90      0.73      0.79     61502



In [14]:
import joblib

joblib.dump(lgb_model, '../outputs/models/lightgbm_final_model.pkl')
print("Model saved")


Model saved
